# 09. Why do M7 crime coefficients approach zero?

This diagnostic reproduces the **primary MPS sample and M6/M7 definitions** from notebook 02. M6 absorbs MSOA and year-quarter effects; M7 absorbs LSOA and year-quarter effects. Both include the same property, location, and economic controls, and use the same transactions.

For each crime category and exposure, we remove the other controls and fixed effects from both log house price and `log(1 + crime)`. A regression of the resulting price residual on the resulting crime residual has exactly the original crime slope (Frisch–Waugh–Lovell theorem). The plots show 20 bins of transaction-level residuals for readability; **the displayed line is fitted to all transactions, not to the bins**.

These plots describe the association used by M6 and M7. They cannot distinguish an absent within-LSOA effect from measurement error, mismatched timing, or remaining confounding.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import qr
from IPython.display import display

ROOT=Path.cwd()
DATA=ROOT/'data'
OUT=DATA/'09_crime_m6_m7_partial_outputs'
OUT.mkdir(parents=True,exist_ok=True)
MERGED=DATA/'crime_housing_merged.csv.gz'
MODELS=DATA/'02_crime_mps_outputs'/'combined_primary_models.csv'
if not MERGED.exists() or not MODELS.exists():
    raise FileNotFoundError('Run the audit and notebook 02 before notebook 09.')
MAIN=['mps_all','mps_violence','mps_property']
EXPOSURES=['annual','lagyear','prior12']
LABELS={'mps_all':'All crime','mps_violence':'Violent crime','mps_property':'Property crime'}
TIMING={'annual':'Current year','lagyear':'Previous year','prior12':'Prior 12 months'}
PROPERTY=['detached_dum','semi_d_dum','terrace_dum','freehold','newbuild']
LOCATION=['tube_distnear_km','bus_distnear_km','thamesriv_dist_km','dist_to_cbd','Nearest_Park_dist_km']
ECONOMIC=['grossannualpay_k','hoursworked','unemployment']
CONTROLS=PROPERTY+LOCATION+ECONOMIC
CRIME_COLS=['log1p_'+crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]
SOURCE_CRIME=[crime+'_'+exposure for exposure in EXPOSURES for crime in MAIN]
raw_columns=['price','age','grossannualpay','Nearest_Park_dist','tube_distnear','bus_distnear',
             'thamesriv_dist','dist_to_cbd','detached_dum','semi_d_dum','terrace_dum','freehold',
             'hoursworked','unemployment','lsoacode','msoa11','oslaua','year','yearquarter',
             'analysis_geography_ok','mps_exposures_available','sale_month']+SOURCE_CRIME
data=pd.read_csv(MERGED,usecols=raw_columns,low_memory=False)
data['lnprice']=np.log(data.price.where(data.price.gt(0)))
data['newbuild']=data.age.eq('Y').astype(float)
data['grossannualpay_k']=data.grossannualpay/1000
data['Nearest_Park_dist_km']=data.Nearest_Park_dist/1000
for source in ['tube_distnear','bus_distnear','thamesriv_dist']:
    data[source+'_km']=data[source]/1000
for source,target in zip(SOURCE_CRIME,CRIME_COLS):
    if data[source].dropna().lt(0).any():
        raise ValueError('Negative crime count: '+source)
    data[target]=np.log1p(data[source])
required=['lnprice','lsoacode','msoa11','oslaua','year','yearquarter']+CONTROLS+CRIME_COLS
valid=data[required].replace('',np.nan).notna().all(axis=1)
valid &= np.isfinite(data[required].select_dtypes(include='number')).all(axis=1)
valid &= data.analysis_geography_ok & data.mps_exposures_available
sample=data.loc[valid,['lnprice','lsoacode','msoa11','year','yearquarter','sale_month']+CONTROLS+CRIME_COLS].copy()
assert sample.sale_month.notna().all()
saved=pd.read_csv(MODELS).query("model in ['M6','M7']").copy()
assert saved.N.nunique()==1 and len(sample)==int(saved.N.iloc[0])
assert sample.lsoacode.nunique()==saved.clusters.iloc[0]
print(f'Common MPS sample: {len(sample):,} sales in {sample.lsoacode.nunique():,} LSOAs.')

Common MPS sample: 338,347 sales in 4,647 LSOAs.


## Partial out controls and fixed effects

All ten outcome/crime columns and the thirteen controls are transformed together under each set of fixed effects. This follows notebook 02's alternating demeaning procedure. We then project each price and crime column off the transformed controls. The crime SD after this second step is the variation actually available to estimate its partial slope. The saved notebook 02 coefficients and its fixed-effect-only crime SD provide independent checks.

In [2]:
def absorb(matrix,groups,tol=1e-10,max_iter=2000):
    z=np.asarray(matrix,dtype=float).copy()
    codes=[pd.factorize(g,sort=False)[0] for g in groups]
    if any((c<0).any() for c in codes):
        raise ValueError('Missing fixed-effect identifier')
    for iteration in range(max_iter):
        before=z.copy()
        for g in codes:
            counts=np.bincount(g)
            sums=np.column_stack([np.bincount(g,weights=z[:,j],minlength=len(counts)) for j in range(z.shape[1])])
            z-=(sums/counts[:,None])[g]
        if np.max(np.abs(z-before))<tol:
            return z,iteration+1
    raise RuntimeError('Fixed-effect absorption did not converge')

summaries=[]
bin_rows=[]
for model,effects in [('M6',['msoa11','yearquarter']),('M7',['lsoacode','yearquarter'])]:
    columns=['lnprice']+CRIME_COLS+CONTROLS
    matrix=sample[columns].to_numpy(float,copy=True)
    scales=matrix.std(axis=0)
    scales[scales<1e-14]=1
    transformed,iterations=absorb(matrix/scales,[sample[c] for c in effects])
    transformed*=scales
    n_crime=len(CRIME_COLS)
    controls=transformed[:,1+n_crime:]
    # Pivoted QR drops fixed-effect-absorbed or collinear control columns.
    Q,R,piv=qr(controls,mode='economic',pivoting=True)
    diagonal=np.abs(np.diag(R))
    rank=int((diagonal>diagonal[0]*1e-10).sum()) if len(diagonal) else 0
    if rank==0:
        raise ValueError('No identified control columns')
    Q=Q[:,:rank]
    targets=transformed[:,:1+n_crime]
    residuals=targets-Q@(Q.T@targets)
    y=residuals[:,0]
    for exposure in EXPOSURES:
        for crime in MAIN:
            name='log1p_'+crime+'_'+exposure
            j=CRIME_COLS.index(name)
            x=residuals[:,1+j]
            beta=float(x@y/(x@x))
            reference=saved.loc[saved.model.eq(model)&saved.crime.eq(crime)&saved.exposure_label.eq(exposure)].iloc[0]
            # Same sample, transformed controls, and FE must reproduce notebook 02.
            if not np.isclose(beta,reference.beta,atol=2e-5,rtol=0):
                raise AssertionError(f'{model} {crime} {exposure}: partial slope {beta} != saved {reference.beta}')
            sd_fe=float(transformed[:,1+j].std(ddof=1))
            if not np.isclose(sd_fe,reference.crime_residual_sd,atol=2e-5,rtol=0):
                raise AssertionError(f'{model} {crime} {exposure}: FE-only crime SD differs')
            sd_full=float(x.std(ddof=1))
            partial_r2=float(np.corrcoef(x,y)[0,1]**2)
            summaries.append(dict(model=model,crime=crime,exposure=exposure,N=len(sample),
                                  lsoa_clusters=sample.lsoacode.nunique(),beta=beta,
                                  saved_beta=reference.beta,beta_difference=beta-reference.beta,
                                  clustered_se=reference.se,clustered_p=reference.p,
                                  crime_sd_raw=float(sample[name].std()),
                                  crime_sd_after_fe=sd_fe,crime_sd_after_fe_and_controls=sd_full,
                                  price_sd_after_fe_and_controls=float(y.std(ddof=1)),
                                  partial_r2=partial_r2,identified_controls=rank,
                                  fe_iterations=iterations))
            # Quantile bins make a very dense transaction-level scatter readable.
            edges=np.unique(np.quantile(x,np.linspace(0,1,21)))
            if len(edges)<3:
                raise ValueError(f'Insufficient partial crime variation: {model} {name}')
            groups=np.clip(np.searchsorted(edges[1:-1],x,side='right'),0,len(edges)-2)
            counts=np.bincount(groups,minlength=len(edges)-1)
            sx=np.bincount(groups,weights=x,minlength=len(edges)-1)
            sy=np.bincount(groups,weights=y,minlength=len(edges)-1)
            for b in np.flatnonzero(counts):
                bin_rows.append(dict(model=model,crime=crime,exposure=exposure,
                                     bin=int(b+1),N=int(counts[b]),
                                     residual_crime_mean=float(sx[b]/counts[b]),
                                     residual_price_mean=float(sy[b]/counts[b])))
    del transformed,residuals,targets,Q,controls,matrix
summary=pd.DataFrame(summaries)
bins=pd.DataFrame(bin_rows)
summary.to_csv(OUT/'partial_regression_summary.csv',index=False)
bins.to_csv(OUT/'partial_regression_bins.csv',index=False)
display(summary[['model','crime','exposure','beta','clustered_p','crime_sd_after_fe',
                 'crime_sd_after_fe_and_controls','partial_r2']])

,model,crime,exposure,beta,clustered_p,crime_sd_after_fe,crime_sd_after_fe_and_controls,partial_r2
0,M6,mps_all,annual,-0.010263,4.105527e-02,0.448230,0.440620,1.710141e-04
1,M6,mps_violence,annual,-0.050159,6.167349e-34,0.510483,0.500521,5.271296e-03
2,M6,mps_property,annual,0.009830,3.833770e-02,0.465132,0.459715,1.707981e-04
3,M6,mps_all,lagyear,-0.008801,8.250470e-02,0.446734,0.439618,1.252095e-04
4,M6,mps_violence,lagyear,-0.048190,4.772170e-34,0.522462,0.512248,5.096238e-03
5,M6,mps_property,lagyear,0.009964,3.915418e-02,0.461242,0.456212,1.728298e-04
6,M6,mps_all,prior12,-0.009672,5.620531e-02,0.447127,0.439870,1.513656e-04
7,M6,mps_violence,prior12,-0.048679,1.473634e-33,0.519040,0.509282,5.140174e-03
8,M6,mps_property,prior12,0.009446,4.891590e-02,0.462255,0.457080,1.558973e-04
9,M7,mps_all,annual,0.005720,2.705113e-01,0.130674,0.130549,5.445999e-06


## Partial regression plots

Each blue point is the mean of roughly one twentieth of the transactions, grouped by residual crime. The orange line has the **transaction-level** M6 or M7 coefficient and passes through the origin of the residualised variables. Bin means are descriptive and are not used to compute the coefficient or p-value. The printed p-value is the original LSOA-clustered p-value from notebook 02.

In [3]:
for exposure in EXPOSURES:
    fig,axes=plt.subplots(2,3,figsize=(16,8),sharex=False,sharey=False)
    for i,model in enumerate(['M6','M7']):
        for j,crime in enumerate(MAIN):
            ax=axes[i,j]
            b=bins.loc[bins.model.eq(model)&bins.crime.eq(crime)&bins.exposure.eq(exposure)]
            s=summary.loc[summary.model.eq(model)&summary.crime.eq(crime)&summary.exposure.eq(exposure)].iloc[0]
            ax.scatter(b.residual_crime_mean,b.residual_price_mean,s=28,color='#2468a0',alpha=.9)
            xlim=np.array([b.residual_crime_mean.min(),b.residual_crime_mean.max()])
            ax.plot(xlim,s.beta*xlim,color='#d07022',linewidth=2,label='Full-sample partial slope')
            ax.axhline(0,color='#888',linewidth=.8)
            ax.axvline(0,color='#888',linewidth=.8)
            ax.set_title(f'{LABELS[crime]} · {model} · β={s.beta:+.4f}, p={s.clustered_p:.3g}')
            ax.set_xlabel('Crime after controls and fixed effects')
            if j==0: ax.set_ylabel('Log price after controls and fixed effects')
            ax.grid(alpha=.2)
    fig.suptitle(f'MPS {TIMING[exposure]}: partial crime–price relationship',fontsize=16)
    fig.tight_layout()
    fig.savefig(OUT/f'partial_regression_{exposure}.png',dpi=170,bbox_inches='tight')
    plt.close(fig)
print('Saved three M6–M7 partial regression figures.')

Saved three M6–M7 partial regression figures.


## Does crime vary within the same LSOA?

This table counts distinct crime exposure values among transactions in each LSOA on the same MPS sample. Annual and previous-year measures can have only a few possible values per LSOA in this short panel; prior-twelve-month values can change by sale month. A change in the crime measure does not itself imply that it explains price changes after controls.

In [4]:
def area_period_crime_values(name,exposure):
    # Annual and previous-year exposure have one value per sale year;
    # prior12 has one rolling-window value per sale month.
    period='sale_month' if exposure=='prior12' else 'year'
    grouped=sample.groupby(['lsoacode',period],sort=True)[name]
    if grouped.nunique().gt(1).any():
        raise ValueError(f'Crime differs within the same LSOA-{period}: {name}')
    return grouped.first()

variation=[]
sales_per_lsoa=sample.groupby('lsoacode').size()
for exposure in EXPOSURES:
    for crime in MAIN:
        name='log1p_'+crime+'_'+exposure
        distinct=sample.groupby('lsoacode')[name].nunique(dropna=True)
        changing=distinct.gt(1)
        variation.append(dict(crime=crime,exposure=exposure,N=len(sample),
                              lsoas=len(distinct),lsoas_with_change=int(changing.sum()),
                              share_lsoas_with_change=float(changing.mean()),
                              share_sales_in_changing_lsoas=float(sales_per_lsoa.loc[changing].sum()/len(sample)),
                              median_distinct_exposure_values=float(distinct.median()),
                              median_lsoa_exposure_sd_equal_periods=float(area_period_crime_values(name,exposure).groupby(level=0).std().fillna(0).median())))
variation=pd.DataFrame(variation)
variation.to_csv(OUT/'within_lsoa_crime_variation.csv',index=False)
display(variation)

wide=summary.pivot(index=['crime','exposure'],columns='model',values=['beta','crime_sd_after_fe_and_controls','partial_r2'])
comparison=pd.DataFrame({
    'crime':wide.index.get_level_values('crime'),
    'exposure':wide.index.get_level_values('exposure'),
    'M6_beta':wide[('beta','M6')].to_numpy(),
    'M7_beta':wide[('beta','M7')].to_numpy(),
    'M6_partial_crime_sd':wide[('crime_sd_after_fe_and_controls','M6')].to_numpy(),
    'M7_partial_crime_sd':wide[('crime_sd_after_fe_and_controls','M7')].to_numpy(),
    'M6_partial_R2':wide[('partial_r2','M6')].to_numpy(),
    'M7_partial_R2':wide[('partial_r2','M7')].to_numpy(),
})
comparison['crime_sd_retained_M7_vs_M6']=comparison.M7_partial_crime_sd/comparison.M6_partial_crime_sd
comparison.to_csv(OUT/'m6_m7_partial_comparison.csv',index=False)
display(comparison)

,crime,exposure,N,lsoas,lsoas_with_change,share_lsoas_with_change,share_sales_in_changing_lsoas,median_distinct_exposure_values,median_lsoa_exposure_sd_equal_periods
0,mps_all,annual,338347,4647,4645,0.999570,0.999867,4.0,0.134976
1,mps_violence,annual,338347,4647,4645,0.999570,0.999840,4.0,0.272725
2,mps_property,annual,338347,4647,4646,0.999785,0.999997,4.0,0.174625
3,mps_all,lagyear,338347,4647,4645,0.999570,0.999988,4.0,0.138161
4,mps_violence,lagyear,338347,4647,4644,0.999354,0.999642,4.0,0.245795
5,mps_property,lagyear,338347,4647,4645,0.999570,0.999988,4.0,0.177617
6,mps_all,prior12,338347,4647,4646,0.999785,0.999997,22.0,0.120972
7,mps_violence,prior12,338347,4647,4646,0.999785,0.999997,14.0,0.226137
8,mps_property,prior12,338347,4647,4646,0.999785,0.999997,20.0,0.154196


,crime,exposure,M6_beta,M7_beta,M6_partial_crime_sd,M7_partial_crime_sd,M6_partial_R2,M7_partial_R2,crime_sd_retained_M7_vs_M6
0,mps_all,annual,-0.010263,0.005720,0.440620,0.130549,0.000171,5.445999e-06,0.296286
1,mps_all,lagyear,-0.008801,0.000767,0.439618,0.128570,0.000125,9.486318e-08,0.292457
2,mps_all,prior12,-0.009672,-0.000586,0.439870,0.129654,0.000151,5.637101e-08,0.294755
3,mps_property,annual,0.009830,0.007391,0.459715,0.157015,0.000171,1.315352e-05,0.341549
4,mps_property,lagyear,0.009964,0.000605,0.456212,0.153515,0.000173,8.411451e-08,0.336499
5,mps_property,prior12,0.009446,-0.000165,0.457080,0.154339,0.000156,6.352077e-09,0.337664
6,mps_violence,annual,-0.050159,0.000635,0.500521,0.240181,0.005271,2.273942e-07,0.479862
7,mps_violence,lagyear,-0.048190,-0.000856,0.512248,0.250913,0.005096,4.506688e-07,0.489828
8,mps_violence,prior12,-0.048679,0.000214,0.509282,0.249921,0.005140,2.805589e-08,0.490732


### Distribution of changes within LSOAs

The next figure uses **every LSOA in the common MPS sample**. For current-year and previous-year crime, it takes one value per observed **sale year** in each LSOA, then calculates the SD across those years with equal weight. For prior-12-month crime, it takes one value per observed **sale month** and weights months equally. It plots the cumulative distribution of these SDs across LSOAs. A line far to the left means many LSOAs have little change over the observed time periods.

This corrects a common pitfall: calculating SD across sales would give more weight to years or months with more transactions, even though those sales share one crime exposure. The time-period SD still does **not** remove year-quarter effects or the other controls; use the fully adjusted SD in `partial_regression_summary.csv` for the variation available to estimate M7.

In [5]:
lsoa_sd_rows=[]
sales_per_lsoa=sample.groupby('lsoacode').size()
for exposure in EXPOSURES:
    for crime in MAIN:
        name='log1p_'+crime+'_'+exposure
        by_period=area_period_crime_values(name,exposure)
        grouped=by_period.groupby(level=0)
        item=pd.DataFrame({'lsoacode':grouped.size().index,
                           'crime':crime,'exposure':exposure,
                           'sales':sales_per_lsoa.loc[grouped.size().index].to_numpy(),
                           'observed_periods':grouped.size().to_numpy(),
                           'distinct_values':grouped.nunique().to_numpy(),
                           'within_lsoa_sd_equal_periods':grouped.std().fillna(0).to_numpy()})
        lsoa_sd_rows.append(item)
lsoa_sd=pd.concat(lsoa_sd_rows,ignore_index=True)
lsoa_sd.to_csv(OUT/'within_lsoa_crime_sd.csv',index=False)

fig,axes=plt.subplots(3,3,figsize=(14,10))
for i,exposure in enumerate(EXPOSURES):
    for j,crime in enumerate(MAIN):
        ax=axes[i,j]
        values=np.sort(lsoa_sd.loc[lsoa_sd.exposure.eq(exposure)&lsoa_sd.crime.eq(crime),
                                    'within_lsoa_sd_equal_periods'].to_numpy())
        ax.plot(values,np.arange(1,len(values)+1)/len(values),color='#2468a0',linewidth=2)
        median=float(np.median(values))
        ax.axvline(median,color='#d07824',linestyle='--',linewidth=1.5,
                   label=f'Median = {median:.3f}')
        ax.set_title(f'{LABELS[crime]} · {TIMING[exposure]}')
        ax.set_xlabel('Crime SD across observed years/months within LSOA')
        if j==0:ax.set_ylabel('Share of LSOAs at or below SD')
        ax.set_ylim(0,1)
        ax.grid(alpha=.2)
        ax.legend(loc='lower right',fontsize=8)
fig.suptitle('How much does crime exposure change over time within LSOAs?',fontsize=16)
fig.tight_layout()
fig.savefig(OUT/'within_lsoa_crime_sd_distribution.png',dpi=170,bbox_inches='tight')
plt.close(fig)
print('Saved equal-period within-LSOA crime SD distributions for all nine MPS measures.')

Saved equal-period within-LSOA crime SD distributions for all nine MPS measures.


## Reading these diagnostics

- A flatter M7 slope means the crime–price association left after LSOA and year-quarter effects is smaller. The plots make that comparison visible; the slope verification checks that they represent notebook 02's regression.
- The crime SD after **fixed effects and other controls** measures the regressor variation available to estimate each coefficient. Less variation can reduce precision, but it does not mechanically shrink the coefficient toward zero.
- A near-zero slope even when partial crime variation remains suggests little linear association in this sample. It does not prove there is no causal effect. Measurement error, exposure timing, heterogeneous effects, and unobserved changes could also matter.
- M6 and M7 use identical transactions, so their coefficient change is not due to sample size. M7 removes time-invariant differences between LSOAs that M6 still uses.
- These are transaction-level diagnostics. Repeated sales share crime exposures, so the bins are visual summaries; use the saved LSOA-clustered inference for formal uncertainty.

### What this run shows

For **prior-12-month MPS crime**, the fully partialled-out results are:

| Crime | M6 coefficient | M7 coefficient | M6 partial crime SD | M7 partial crime SD |
|---|---:|---:|---:|---:|
| All crime | −0.0097 | −0.0006 | 0.4399 | 0.1297 |
| Violent crime | −0.0487 | +0.0002 | 0.5093 | 0.2499 |
| Property crime | +0.0094 | −0.0002 | 0.4571 | 0.1543 |

M7 retains about 29% of M6's partial crime SD for all crime, 49% for violent crime, and 34% for property crime. Its coefficient is nearly zero despite that remaining variation. Nearly every observed LSOA has more than one crime-exposure value in this sample, but the number of distinct values and their reliability differ by exposure window. This pattern is **consistent with** M6 using persistent differences between LSOAs that M7 absorbs. It does not establish that these differences cause the M6 association or identify the true crime effect. The complete results for all nine combinations are in `m6_m7_partial_comparison.csv`.


For the current-year and previous-year exposures, the MPS sales sample covers **four sale years (2012–2015)**. Thus an LSOA has at most four observed year-level values in this sample, and the median has four distinct values. Previous-year exposures refer to crime in 2011–2014. The `within_lsoa_crime_sd.csv` and distribution plot give each observed year equal weight for these two exposures. Prior12 is month-specific and uses equal weights across observed sale months. The older sale-weighted SD plot was replaced by this period-weighted version.


### Does the remaining variation let us estimate M7?

Yes, although its identifying variation is smaller. For prior-12-month violent crime, 4,646 of 4,647 observed LSOAs have more than one crime-exposure value and the median LSOA has 14 distinct values among its sales. Its crime SD after fixed effects **and controls** is 0.2499 in M7, compared with 0.5093 in M6. M7's crime coefficient is +0.0002 with LSOA-clustered SE 0.0030 and 95% CI about −0.0056 to +0.0060. The M6 estimate of −0.0487 lies far outside that M7 interval. The near-zero M7 estimate therefore cannot be explained merely by an inability to calculate a slope or by all crime variation disappearing.

Less within-LSOA variation may still weaken precision, and measurement error can become more consequential relative to the true within-LSOA signal. Standard errors also depend on residual price variation and within-cluster dependence; they need not rise when crime SD falls. In the prior-12-month models, the M6-to-M7 clustered SE falls for violent crime (0.0040 to 0.0030), rises for all crime (0.0051 to 0.0061), and changes little for property crime (0.0048 to 0.0047). The raw within-LSOA SD distribution is **not** a measure of the variation left after year-quarter effects and controls; use `partial_regression_summary.csv` for that adjusted measure.
